# Movie Recommendation System

Build an item-to-item collaborative-filtering recommender from explicit movie ratings.

**Portfolio category:** Recommendation

**Data mode:** Committed dataset

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

Use only portable paths and keep ratings separate from evaluation logic.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. Data loading

The committed tables follow a compact MovieLens-style schema.

In [ ]:
project_dir = Path.cwd()
if not (project_dir / "ratings.tsv").exists():
    project_dir = Path("Unsupervised Learning Projects/Movie Recommendation System")

ratings = pd.read_csv(
    project_dir / "ratings.tsv",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
)
titles = pd.read_csv(project_dir / "movie_titles.csv")
interactions = ratings.merge(titles, on="item_id", how="inner")
interactions.head()

## 3. Data quality and behaviour checks

In [ ]:
quality = pd.Series({
    "ratings": len(interactions),
    "users": interactions["user_id"].nunique(),
    "movies": interactions["item_id"].nunique(),
    "missing_cells": int(interactions.isna().sum().sum()),
    "duplicate_user_movie": int(interactions.duplicated(["user_id", "item_id"]).sum()),
})
display(quality.to_frame("value"))
display(interactions["rating"].describe().to_frame().T)

## 4. Exploratory analysis

In [ ]:
movie_stats = interactions.groupby("title").agg(
    rating_count=("rating", "size"),
    mean_rating=("rating", "mean"),
).sort_values("rating_count", ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(interactions["rating"], discrete=True, ax=axes[0])
movie_stats.head(15).sort_values("rating_count").plot.barh(
    y="rating_count", legend=False, ax=axes[1], color="#2563eb"
)
axes[0].set_title("Rating distribution")
axes[1].set_title("Most-rated movies")
plt.tight_layout()

## 5. Interaction filtering and similarity model

In [ ]:
min_movie_ratings = max(20, int(movie_stats["rating_count"].quantile(0.65)))
active_movies = movie_stats.query("rating_count >= @min_movie_ratings").index
filtered = interactions[interactions["title"].isin(active_movies)]
user_movie = filtered.pivot_table(index="user_id", columns="title", values="rating")
movie_user = user_movie.T.fillna(0)
similarity = pd.DataFrame(
    cosine_similarity(movie_user),
    index=movie_user.index,
    columns=movie_user.index,
)
np.fill_diagonal(similarity.values, 0)

def recommend(title, n=8):
    neighbours = similarity.loc[title].nlargest(n)
    return pd.DataFrame({"title": neighbours.index, "similarity": neighbours.values})

seed_title = movie_stats.loc[active_movies].sort_values("rating_count", ascending=False).index[0]
recommendations = recommend(seed_title)
print(f"Seed movie: {seed_title}")
display(recommendations)

## 6. Evaluation without pretending similarity is accuracy

In [ ]:
sample_titles = similarity.index[: min(100, len(similarity))]
top_neighbours = {
    title: set(similarity.loc[title].nlargest(5).index)
    for title in sample_titles
}
recommended_catalogue = set().union(*top_neighbours.values())
coverage = len(recommended_catalogue) / len(similarity)
mean_top_similarity = np.mean([
    similarity.loc[title].nlargest(5).mean() for title in sample_titles
])
display(pd.DataFrame({
    "metric": ["catalogue coverage", "mean top-5 similarity"],
    "value": [coverage, mean_top_similarity],
}))

## 7. Latent-space diagnostic

In [ ]:
n_components = min(12, movie_user.shape[1] - 1, movie_user.shape[0] - 1)
embedding = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE).fit_transform(movie_user)
sns.scatterplot(x=embedding[:, 0], y=embedding[:, 1], alpha=0.6)
plt.title("Movies in a latent interaction space")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.tight_layout()

## 8. Recommendation inspection

In [ ]:
inspection = recommendations.merge(movie_stats, left_on="title", right_index=True)
display(inspection.round(3))

## 9. Key findings

Popularity filtering reduces noisy similarities; catalogue coverage reveals whether recommendations collapse onto a few titles.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For movie recommendation system,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.